In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

data_dir = Path("../data")
files = sorted(data_dir.rglob("raw_*.csv"))

print(f"Found {len(files)} asset files\n")

series_list = []

for path in files:
    asset = path.stem.replace("raw_", "")

    df = pd.read_csv(path, usecols=["Date", "Close"])
    df["Date"] = pd.to_datetime(df["Date"])
    df = df.sort_values("Date")

    s = df.set_index("Date")["Close"]

    n_negative = (s < 0).sum()
    if n_negative > 0:
        print(f"  [{asset}] {n_negative} negative price(s) detected — forward-filled")

    s = s.where(s > 0, np.nan).ffill()
    log_returns = np.log(s / s.shift(1)).rename(asset)

    series_list.append(log_returns)

print(f"\nLoaded {len(series_list)} assets")

merged = pd.concat(series_list, axis=1, sort=False)
print(f"Merged shape before dropna: {merged.shape}")
print(f"Total NaNs before dropna: {merged.isna().sum().sum()}")

merged = merged.dropna()
print(f"Merged shape after dropna:  {merged.shape}")

merged = merged.reset_index()
merged = merged.sort_values("Date").reset_index(drop=True)

print(f"\nDate range: {merged['Date'].min()} → {merged['Date'].max()}")
print(f"Assets: {[c for c in merged.columns if c != 'Date']}")

merged["Date"] = merged["Date"].dt.strftime("%Y-%m-%d")
merged.to_csv(data_dir / "all_assets_log_returns.csv", index=False)

print(f"\nSaved → {data_dir / 'all_assets_log_returns.csv'}")
print(f"Final shape: {merged.shape[0]} trading days × {merged.shape[1] - 1} assets (+1 Date column)")

merged.head()
